# Learning Eigenmodes using U-Nets

This time, we train Eigenmodes using U-Nets. The dataset preparation and all that is the same as FNOs

In [7]:
import torch
import torch.nn as nn
import os
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
from torch.optim import Adam
import matplotlib.pyplot as plt
from LightPipes import *

import unet
from unet import UNet

import fno
from fno import overlap_loss


Load the dataset

In [8]:
NPZ_PATH = "datasets/dataset2.npz"   # ← path to the pre-converted .npz file

npz = np.load(NPZ_PATH)
data = {k: npz[k] for k in npz.files}

print(f"Loaded dataset from '{NPZ_PATH}'")
print(f"Keys    : {list(data.keys())}")
print(f"Samples : {data['gaussian_forward_real'].shape[0]}")
print(f"Grid    : {data['gaussian_forward_real'].shape[1:]} (H x W)")

Loaded dataset from 'datasets/dataset2.npz'
Keys    : ['eigenmode_1_real', 'eigenmode_2_real', 'eigenmode_3_real', 'eigenmode_4_real', 'eigenmode_1_imag', 'eigenmode_2_imag', 'eigenmode_3_imag', 'eigenmode_4_imag', 'gaussian_forward_real', 'gaussian_forward_imag', 'gaussian_reversed_real', 'gaussian_reversed_imag']
Samples : 10001
Grid    : (64, 64) (H x W)


Prepare the training dataset

In [9]:
N_EIGENMODES = 1

def normalize_11(arr: np.ndarray) -> np.ndarray:
    """Min-max normalise arr to the range [-1, +1]."""
    lo, hi = arr.min(), arr.max()
    if hi - lo < 1e-12:          # constant array — avoid division by zero
        return np.zeros_like(arr)
    return 2.0 * (arr - lo) / (hi - lo) - 1.0

def prepare_pairs(data, n_eigenmodes=N_EIGENMODES):
    """
    Build single-step predictor pairs from the dataset.

    For each consecutive pair of timesteps (t, t+1) and each eigenmode k we form:
      - Input  : [eigenmode_k_real_t,        eigenmode_k_imag_t,
                  gaussian_forward_real_t+1,  gaussian_forward_imag_t+1,
                  gaussian_reversed_real_t+1, gaussian_reversed_imag_t+1]  → (H, W, 6)
      - Target : [eigenmode_k_real_{t+1}, eigenmode_k_imag_{t+1}]          → (H, W, 2)

    Samples are interleaved by timestep (outer) then eigenmode (inner), matching
    prepare_pairs_slow, so that a simple train/test slice gives a proper temporal holdout:
      row index = t * n_eigenmodes + (k - 1)

    The Gaussian forward/reversed channels are normalised to [-1, +1].

    Returns
    -------
    X : torch.Tensor  shape ((N-1)*n_eigenmodes, H, W, 6)
    Y : torch.Tensor  shape ((N-1)*n_eigenmodes, H, W, 2)
    """
    N_samples = data["gaussian_forward_real"].shape[0]
    H, W      = data["gaussian_forward_real"].shape[1:]
    N_pairs   = N_samples - 1
    total     = N_pairs * n_eigenmodes

    # Pre-allocate output arrays
    X = np.empty((total, H, W, 6), dtype=np.float32)
    Y = np.empty((total, H, W, 2), dtype=np.float32)

    # Load and normalise Gaussian arrays once — shared across all eigenmodes
    gfwd_real = normalize_11(data["gaussian_forward_real"][1:].astype(np.float32))  # (N-1, H, W)
    gfwd_imag = normalize_11(data["gaussian_forward_imag"][1:].astype(np.float32))
    grev_real = normalize_11(data["gaussian_reversed_real"][1:].astype(np.float32))
    grev_imag = normalize_11(data["gaussian_reversed_imag"][1:].astype(np.float32))

    # Load all eigenmode arrays up front
    em = {}
    for k in range(1, n_eigenmodes + 1):
        em[k] = (data[f"eigenmode_{k}_real"].astype(np.float32),
                 data[f"eigenmode_{k}_imag"].astype(np.float32))

    # Fill in timestep-major order: row = t * n_eigenmodes + (k-1)
    for t in range(N_pairs):
        for k in range(1, n_eigenmodes + 1):
            row = t * n_eigenmodes + (k - 1)
            em_real, em_imag = em[k]

            X[row, ..., 0] = em_real[t]       # eigenmode k at t
            X[row, ..., 1] = em_imag[t]
            X[row, ..., 2] = gfwd_real[t]     # Gaussian forward at t+1  (normalised)
            X[row, ..., 3] = gfwd_imag[t]
            X[row, ..., 4] = grev_real[t]     # Gaussian reversed at t+1 (normalised)
            X[row, ..., 5] = grev_imag[t]

            Y[row, ..., 0] = em_real[t + 1]   # eigenmode k at t+1
            Y[row, ..., 1] = em_imag[t + 1]

        if (t + 1) % 50 == 0 or (t + 1) == N_pairs:
            print(f"  Processed {t+1}/{N_pairs} timestep pairs ...")

    return torch.from_numpy(X), torch.from_numpy(Y)


X, Y = prepare_pairs(data)

print(f"Input  shape : {X.shape}")   # ((N-1)*4, H, W, 6)
print(f"Target shape : {Y.shape}")   # ((N-1)*4, H, W, 2)


  Processed 50/10000 timestep pairs ...
  Processed 100/10000 timestep pairs ...
  Processed 150/10000 timestep pairs ...
  Processed 200/10000 timestep pairs ...
  Processed 250/10000 timestep pairs ...
  Processed 300/10000 timestep pairs ...
  Processed 350/10000 timestep pairs ...
  Processed 400/10000 timestep pairs ...
  Processed 450/10000 timestep pairs ...
  Processed 500/10000 timestep pairs ...
  Processed 550/10000 timestep pairs ...
  Processed 600/10000 timestep pairs ...
  Processed 650/10000 timestep pairs ...
  Processed 700/10000 timestep pairs ...
  Processed 750/10000 timestep pairs ...
  Processed 800/10000 timestep pairs ...
  Processed 850/10000 timestep pairs ...
  Processed 900/10000 timestep pairs ...
  Processed 950/10000 timestep pairs ...
  Processed 1000/10000 timestep pairs ...
  Processed 1050/10000 timestep pairs ...
  Processed 1100/10000 timestep pairs ...
  Processed 1150/10000 timestep pairs ...
  Processed 1200/10000 timestep pairs ...
  Processed 

Prepare the dataset in a different way

In [10]:
def prepare_pairs_delta(data, n_eigenmodes=N_EIGENMODES, include_lag = False, lag=1):
    """
    Build single-step *delta* predictor pairs with extended probe context.

    For each consecutive pair (t → t+1), with t starting from `lag` to allow
    access to t-lag, and each eigenmode k the input has 11 channels:

      ch  0,1  : norm(em_k_real_t),    norm(em_k_imag_t)    eigenmode at t  (normalised)
      ch  2,3  : P→(t)   Re/Im  = gaussian_forward[t]        current forward probe
      ch  4,5  : P←(t)   Re/Im  = gaussian_reversed[t]       current reverse probe
      ch  6,7  : P→(t-N) Re/Im  = gaussian_forward[t-lag]    lagged forward probe
      ch  8,9  : P←(t-N) Re/Im  = gaussian_reversed[t-lag]   lagged reverse probe
      ch 10    : ΔI→(t,N) = I→(t) − I→(t-lag)               forward frame difference

    Target (2 channels):
      normalised delta  norm(em_k_{t+1}) − norm(em_k_t)

    Parameters
    ----------
    lag : int
        Number of frames to look back for the lagged probe channels and frame
        difference (default 1, i.e. t-1).  The loop starts at t=lag so that
        t-lag is always a valid index, yielding (N_samples - lag - 1) pairs.

    Returns
    -------
    X : torch.Tensor  shape ((N-lag-1)*n_eigenmodes, H, W, 11)
    Y : torch.Tensor  shape ((N-lag-1)*n_eigenmodes, H, W,  2)  ← normalised residuals
    """
    N_samples = data["gaussian_forward_real"].shape[0]
    H, W      = data["gaussian_forward_real"].shape[1:]
    N_pairs   = N_samples - lag - 1   # t runs from lag to N_samples-2 inclusive
    total     = N_pairs * n_eigenmodes

    if (include_lag):
        X = np.empty((total, H, W, 11), dtype=np.float32)
    else: 
        X = np.empty((total, H, W, 6), dtype=np.float32)

    Y = np.empty((total, H, W,  2), dtype=np.float32)

    # ── Load all Gaussian probe arrays (full time axis) ───────────────────────
    gfwd_real = data["gaussian_forward_real"].astype(np.float32)    # (N, H, W)
    gfwd_imag = data["gaussian_forward_imag"].astype(np.float32)
    grev_real = data["gaussian_reversed_real"].astype(np.float32)
    grev_imag = data["gaussian_reversed_imag"].astype(np.float32)

    # ── Forward intensity I→(t) = |P→(t)|²  ──────────────────────────────────
    fwd_intensity = gfwd_real ** 2 + gfwd_imag ** 2                 # (N, H, W)

    # ── Eigenmode arrays: raw for delta targets, normalised for inputs ────────
    em_raw  = {}
    em_norm = {}
    for k in range(1, n_eigenmodes + 1):
        raw_r = data[f"eigenmode_{k}_real"].astype(np.float32)
        raw_i = data[f"eigenmode_{k}_imag"].astype(np.float32)
        em_raw[k]  = (raw_r, raw_i)
        em_norm[k] = (normalize_11(raw_r), normalize_11(raw_i))

    # ── Fill arrays — t runs from lag to N_samples-2 ─────────────────────────
    for t in range(lag, N_samples - 1):
        pair_idx = t - lag   # 0-based row offset
        for k in range(1, n_eigenmodes + 1):
            row = pair_idx * n_eigenmodes + (k - 1)
            raw_r,  raw_i  = em_raw[k]
            norm_r, norm_i = em_norm[k]

            # ch 0,1 — normalised eigenmode at t
            X[row, ..., 0] = norm_r[t]
            X[row, ..., 1] = norm_i[t]

            # ch 2,3 — current forward probe P→(t)
            X[row, ..., 2] = gfwd_real[t]
            X[row, ..., 3] = gfwd_imag[t]

            # ch 4,5 — current reverse probe P←(t)
            X[row, ..., 4] = grev_real[t]
            X[row, ..., 5] = grev_imag[t]

            if (include_lag): # Include previous Gaussian probes to get temporal data
                # ch 6,7 — lagged forward probe P→(t-N)
                X[row, ..., 6] = gfwd_real[t - lag]
                X[row, ..., 7] = gfwd_imag[t - lag]

                # ch 8,9 — lagged reverse probe P←(t-N)
                X[row, ..., 8] = grev_real[t - lag]
                X[row, ..., 9] = grev_imag[t - lag]

                # ch 10 — forward frame difference ΔI→(t,N) = I→(t) − I→(t-N)
                X[row, ..., 10] = fwd_intensity[t] - fwd_intensity[t - lag]

            # target — normalised delta
            Y[row, ..., 0] = norm_r[t + 1] - norm_r[t]
            Y[row, ..., 1] = norm_i[t + 1] - norm_i[t]

        if t % 50 == 0 or t == N_samples - 2:
            print(f"  Processed {t - lag + 1}/{N_pairs} timestep pairs ...")

    return torch.from_numpy(X), torch.from_numpy(Y)


def reconstruct_from_delta(
    input_sample: torch.Tensor,
    delta_pred: torch.Tensor,
) -> torch.Tensor:
    """
    Reconstruct the predicted next eigenmode from the input and a predicted delta.

    Parameters
    ----------
    input_sample : torch.Tensor  (..., 2)  — eigenmode at t
    delta_pred   : torch.Tensor  (..., 2)  — predicted ΔE

    Returns
    -------
    torch.Tensor  (..., 2)  — predicted eigenmode at t+1
    """
    return input_sample + delta_pred


# ── Example usage ──────────────────────────────────────────────────────────────
LAG = 2  # ← set the desired lag N here
X, Y = prepare_pairs_delta(data, lag=LAG)

print(f"Lag            : {LAG} frames")
print(f"Delta input  shape : {X.shape}")   # ((N-lag-1)*n_eigenmodes, H, W, 11)
print(f"Delta target shape : {Y.shape}  ← normalised residuals ΔE")

  Processed 49/9998 timestep pairs ...
  Processed 99/9998 timestep pairs ...
  Processed 149/9998 timestep pairs ...
  Processed 199/9998 timestep pairs ...
  Processed 249/9998 timestep pairs ...
  Processed 299/9998 timestep pairs ...
  Processed 349/9998 timestep pairs ...
  Processed 399/9998 timestep pairs ...
  Processed 449/9998 timestep pairs ...
  Processed 499/9998 timestep pairs ...
  Processed 549/9998 timestep pairs ...
  Processed 599/9998 timestep pairs ...
  Processed 649/9998 timestep pairs ...
  Processed 699/9998 timestep pairs ...
  Processed 749/9998 timestep pairs ...
  Processed 799/9998 timestep pairs ...
  Processed 849/9998 timestep pairs ...
  Processed 899/9998 timestep pairs ...
  Processed 949/9998 timestep pairs ...
  Processed 999/9998 timestep pairs ...
  Processed 1049/9998 timestep pairs ...
  Processed 1099/9998 timestep pairs ...
  Processed 1149/9998 timestep pairs ...
  Processed 1199/9998 timestep pairs ...
  Processed 1249/9998 timestep pairs .

Build train/test pairs

In [11]:
n_total = int(0.50*X.shape[0])
n_train = int(0.85*n_total)
batch_size = 10

x_train =  X[:n_train]
x_test =  X[n_train:n_total]

y_train = Y[:n_train]
y_test = Y[n_train:n_total]

N = x_train.shape[2]

training_set = DataLoader(TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True)
testing_set = DataLoader(TensorDataset(x_test, y_test), batch_size=batch_size, shuffle=False)

Instantiate the U-Net Architecture

In [12]:
epochs = 50

# These control the step-wise updates on the LR

learn_rate = 0.001  # initial LR
decay_rate = 0.001  # initial weight decay
step_size  = 12     # epochs between LR decay steps
gamma      = 0.5    # LR decay factor

# U-Net architecture

nnType = 0
kernelSize = 3
layers = 1
dropRate = 0.1

# Physical parameters 
size = 0.45
lensSize = 0.1125

Load up the GPU (if not already done so)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Is CUDA available? {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

print(f"Using device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() el

Apply aperture to output channel 

In [ ]:
def torch_circ_aperture(field_batch: torch.Tensor, size: float, lensSize: float) -> torch.Tensor:
    """
    Applies a circular aperture to a batch of complex fields represented
    as real/imaginary channels.

    Parameters
    ----------
    field_batch : torch.Tensor  shape (batch, H, W, C)
                  where C=2 holds [real, imag] channels
    size        : float — physical side length of the window (metres)
    lensSize    : float — aperture radius (metres)

    Returns
    -------
    torch.Tensor — same shape as field_batch, zeroed outside the aperture
    """
    _, H, W, _ = field_batch.shape
    device = field_batch.device

    # Build physical coordinate axes centred at zero — same convention as LightPipes
    x = torch.linspace(-size / 2, size / 2, W, device=device)
    y = torch.linspace(-size / 2, size / 2, H, device=device)
    yy, xx = torch.meshgrid(y, x, indexing="ij")   # (H, W)

    # Binary mask: 1 inside aperture, 0 outside
    mask = (xx**2 + yy**2 <= lensSize**2).float()  # (H, W)
    mask = mask.unsqueeze(0).unsqueeze(-1)          # (1, H, W, 1) — broadcasts over batch and channels

    return field_batch * mask


Instantiate and train the UNet. Use the fidelity as a loss metric

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

u_net = UNet(64, nnType, layers=layers)
optim = Adam(UNet.parameters(), lr=learn_rate, weight_decay=decay_rate)
scheduler = torch.optim.lr_scheduler.StepLR(optim, step_size=step_size, gamma=gamma)

loss = overlap_loss
freq_print = 1

n_train_batches = len(training_set)
n_test_batches = len(testing_set)

train_mse_history = []
test_rel_l2_history = []

for epoch in range(epochs):

    # Evaluate the training metric
    u_net.train()
    train_mse = 0.0

    for step, (input_batch, delta_batch) in enumerate(training_set):
        input_batch = input_batch.to(device)   # (B, H, W, 6)
        delta_batch = delta_batch.to(device)   # (B, H, W, 2) — true ΔE

        optim.zero_grad()

        # Predict ΔE and apply aperture
        delta_pred_batch = torch_circ_aperture(fno(input_batch), size=size, lensSize=lensSize)  # (B, H, W, 2)

        # Reconstruct E_{t+1} = E_t + ΔE  (channels 0,1 of input are the eigenmode at t)
        output_pred_batch = reconstruct_from_delta(input_batch[..., :2], delta_pred_batch)   # (B, H, W, 2)
        output_batch      = reconstruct_from_delta(input_batch[..., :2], delta_batch)        # (B, H, W, 2)

        loss_f = loss(output_pred_batch, output_batch)
        loss_f.backward()
        optim.step()
        train_mse += loss_f.item()

    train_mse /= n_train_batches
    print("Training batches computed")

    scheduler.step()

    with torch.no_grad():
        fno.eval()
        test_relative_l2 = 0.0
        for step, (input_batch, delta_batch) in enumerate(testing_set):
            input_batch = input_batch.to(device)   # (B, H, W, 6)
            delta_batch = delta_batch.to(device)   # (B, H, W, 2) — true ΔE

            # Predict ΔE and apply aperture
            delta_pred_batch = torch_circ_aperture(fno(input_batch), size=size, lensSize=lensSize)  # (B, H, W, 2)

            # Reconstruct E_{t+1} = E_t + ΔE
            output_pred_batch = reconstruct_from_delta(input_batch[..., :2], delta_pred_batch)   # (B, H, W, 2)
            output_batch      = reconstruct_from_delta(input_batch[..., :2], delta_batch)        # (B, H, W, 2)
            loss_f = loss(output_pred_batch, output_batch)

            test_relative_l2 += loss_f.item()

        test_relative_l2 /= n_test_batches

    train_mse_history.append(train_mse)
    test_rel_l2_history.append(test_relative_l2)

    if epoch % freq_print == 0:
        print(f"========= Epoch {epoch+1}/{epochs} Summary ========= Train Loss: {train_mse:.6f} | Mean Test Loss: {test_relative_l2:.4f}\n")
